# F02-P2 Climate

**Site Characterization: Climate, components 3.x.**

Quantifies the carbon side of the site and describes its bioclimatic setting. Where General
Context and Nature describe the site, Climate puts numbers on the climate value of it.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** 3.1 Current Carbon Storage, 3.2 Soil Organic Carbon, 3.3 Annual Temperature and
3.4 Annual Precipitation are written. Later components are not started.

## Two separate carbon headline numbers, deliberately not summed

3.1 reports biomass carbon and 3.2 reports soil carbon, each as its own big number. The team
decided not to publish a combined total. The reason to keep in mind when reading them: adding
the two would still not be total site carbon, because deadwood and litter are absent from both,
and because the soil layer only reaches a fixed depth. A combined figure would look like a
complete account when it is not.

## Handoff

Optionally reads `outputs/<aoi_id>__F02-P2-general.json` to learn whether the AOI contains
peatland, which changes how 3.2 should be read. The notebook runs without it; the peat check is
skipped and a flag records that. Writes `outputs/<aoi_id>__F02-P2-climate.json`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass, replace

import geopandas as gpd
import numpy as np

from config import *
from common import *

In [ ]:
AOI_PATH = r"<SET: path to the project AOI polygon>"
aoi_id = "<SET: short id for this run, must match the other F02-P2 notebooks>"

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}")

---
## 3.1 Current Carbon Storage

Reports how much carbon the project area holds right now, as a single headline number in tonnes
of CO2 equivalent, with a breakdown per carbon pool in tCO2e and in percent.

**Data.** Two continuous rasters holding **dry biomass density in Mg/ha**, not carbon:

- `agb_mgha.tif` - aboveground biomass. The in-house layer, Alpha Earth plus GEDI.
- `bgb_mgha.tif` - belowground biomass, that is roots.

**What this number covers, and what it does not.** Two of the five IPCC carbon pools are
included: aboveground biomass and belowground biomass. Deadwood, litter and **soil organic
carbon are all excluded**.

The soil exclusion matters most on peat. In tropical peat swamp forest the biomass pools
typically hold only a small share of total site carbon, often in the region of 5 to 15 percent,
because the peat itself can store on the order of thousands of tCO2e per hectare depending on
depth. A peatland AOI will therefore report a headline that is far below its real stock. Since
Peatland is one of the three Axis 3 classes in 1.1, this case will occur. The component
therefore labels its output as biomass carbon rather than total carbon, and carries the pool
list in `values` so the frontend can state the scope next to the big number.

**Read the pool split with care.** If `bgb_mgha.tif` was produced upstream by multiplying AGB by
a fixed root to shoot ratio, then the AGB and BGB shares are constant by construction: a ratio
of 0.25 always yields an 80 / 20 split, on every site. The percentages would then carry no site
specific information and should not be presented as a finding. The split is only informative if
BGB was mapped independently, or if the root to shoot ratio varies by ecosystem or biomass
class. This needs one check against how the BGB layer was built.

**The two conversions, kept visible in the tool.**

```
carbon_tC    = biomass_Mg * CARBON_FRACTION      # 0.47, IPCC 2006 GL Vol 4 Ch 4
storage_tCO2e = carbon_tC * CO2_PER_C            # 44 / 12 = 3.667
```

Neither step is done upstream. Reading the biomass rasters as biomass and applying both factors
here means the assumptions live in `config.py` where they can be audited and changed, rather
than being baked into a raster that looks like a measurement.

**Decisions locked.**

- Extent is every valid pixel of the biomass rasters inside the AOI, not forest only. The
  headline claims the stock of the site, so shrubland, agroforestry and plantation biomass count.
- Nodata inside the AOI is treated as zero biomass. Accepted by the team. The component still
  measures raster coverage and flags an AOI where coverage falls below
  `CARBON_COVERAGE_WARN_PCT`, because with nodata read as zero an incomplete raster produces a
  quiet under-estimate that the number itself cannot reveal.
- The two pools are integrated separately and then added, so AGB and BGB may sit on different
  grids or resolutions without any alignment assumption.
- Resampling is `average`, not `bilinear`. This is a stock quantity, so the resampling has to
  preserve the area weighted mean when the raster is reprojected to the reference CRS.
- Pool shares are a percentage of the biomass total reported here, not of total site carbon.
  With soil excluded, a share of the total is not a share of the site.

**Example render.**

> **1,284,000 tCO2e**
>
> This project area currently stores approximately 1,284,000 tCO2e in aboveground and
> belowground biomass, an average of 1,036 tCO2e per hectare. Aboveground biomass holds
> 1,027,200 tCO2e (80%) and belowground biomass 256,800 tCO2e (20%). Soil organic carbon is not
> included.

| Carbon pool | Storage (tCO2e) | Share of biomass carbon |
|---|---|---|
| Aboveground biomass | 1,027,200 | 80% |
| Belowground biomass | 256,800 | 20% |
| **Total** | **1,284,000** | **100%** |

**Narrative not yet specified.** The wording above is a placeholder written to make the units
and the scope explicit. Replace it once the team settles the phrasing.

**Downstream use.** The current stock is the reference point for Benefit Quantification in
F02-P5: avoided loss is measured against what is standing, and removals are measured as growth
towards a reference stock. The pool split matters there because the two pools behave differently
under disturbance: aboveground biomass is lost quickly in a clearing event, while root carbon
decays over years.

In [ ]:
CARBON_POOLS = ("Aboveground biomass", "Belowground biomass")


@dataclass(frozen=True)
class CarbonPool:
    """One biomass pool integrated over the AOI."""

    name: str
    biomass_mg: float      # total dry biomass, tonnes
    storage_tco2e: float   # after carbon fraction and 44/12
    coverage_pct: float    # share of the AOI with a valid pixel
    pct: float = 0.0       # share of the biomass carbon total, filled in once both pools exist


def _integrate_pool(name: str, path: str, aoi: AOI) -> CarbonPool:
    """Integrate one biomass raster over the AOI and convert to tCO2e.

    Density times area, so the pixel area cancels the per hectare unit:
        Mg/ha * ha = Mg
    Each pool is integrated on its own grid, so AGB and BGB need not share a resolution.
    """
    raster = load_raster_clipped(path, aoi, resampling="average")

    # Team decision: nodata inside the AOI counts as zero biomass. Coverage is measured
    # separately so an incomplete raster is still visible.
    values = raster.values.filled(0.0).astype(float)

    biomass_mg = float(values.sum()) * raster.pixel_area_ha
    storage_tco2e = biomass_mg * CARBON_FRACTION * CO2_PER_C

    return CarbonPool(
        name=name,
        biomass_mg=biomass_mg,
        storage_tco2e=storage_tco2e,
        coverage_pct=safe_pct(raster.valid_area_ha, aoi.area_ha),
    )


def analyze_current_carbon_storage(aoi: AOI) -> ComponentResult:
    """Component 3.1. Biomass carbon currently stored in the project area, in tCO2e."""
    pools = [
        _integrate_pool(CARBON_POOLS[0], AGB_RASTER, aoi),
        _integrate_pool(CARBON_POOLS[1], BGB_RASTER, aoi),
    ]

    total_tco2e = sum(p.storage_tco2e for p in pools)

    if total_tco2e <= 0:
        return not_applicable(
            "3.1 Current Carbon Storage",
            "No biomass data is available for this project area, so current carbon storage "
            "cannot be estimated.",
        )

    # Shares are of the biomass total reported here, not of total site carbon. Soil is excluded,
    # so these percentages sum to 100 of a partial accounting.
    pools = [replace(p, pct=safe_pct(p.storage_tco2e, total_tco2e)) for p in pools]

    density_tco2e_ha = total_tco2e / aoi.area_ha if aoi.area_ha > 0 else 0.0
    coverage_pct = max(p.coverage_pct for p in pools)

    flags: list[str] = []
    if coverage_pct < CARBON_COVERAGE_WARN_PCT:
        flags.append(
            f"3.1: the biomass rasters cover only {coverage_pct:.0f}% of the AOI. Nodata is "
            "counted as zero biomass, so the headline is an under-estimate by an unknown "
            "amount."
        )

    # Placeholder wording, see the note above.
    breakdown = oxford_join(
        f"{p.name.lower()} holds {p.storage_tco2e:,.0f} tCO2e ({fmt_pct(p.pct)})"
        for p in pools
    )
    narrative = (
        f"This project area currently stores approximately {total_tco2e:,.0f} tCO2e in "
        f"aboveground and belowground biomass, an average of {density_tco2e_ha:,.0f} tCO2e per "
        f"hectare. Of this, {breakdown}. Soil organic carbon is not included."
    )

    return ComponentResult(
        component="3.1 Current Carbon Storage",
        applicable=True,
        narrative=narrative,
        tables={"pools": pools},  # name, storage_tco2e, pct -> breakdown table
        values={
            "total_tco2e": total_tco2e,          # headline big number
            "density_tco2e_ha": density_tco2e_ha,
            "coverage_pct": coverage_pct,
            "pool_tco2e": {p.name: p.storage_tco2e for p in pools},
            "pool_pct": {p.name: p.pct for p in pools},
            "pools_included": list(CARBON_POOLS),
            "pools_excluded": ["deadwood", "litter", "soil organic carbon"],
        },
        flags=flags,
    )

---
## 3.2 Soil Organic Carbon

Reports the organic carbon held in the soil of the project area, as a single headline number in
tonnes of CO2 equivalent.

**Data.** `soil_carbon.tif`, soil organic carbon stock in **tC/ha**, carbon rather than biomass,
representing a fixed depth of `SOIL_CARBON_DEPTH_CM` = 30 cm.

**One conversion.**

```
storage_tCO2e = soc_tC * CO2_PER_C            # 44 / 12 = 3.667
```

No carbon fraction here. Unlike the biomass rasters in 3.1, this layer already holds carbon, so
applying 0.47 would be a double conversion and would understate the stock by roughly half.

**Depth is part of the measurement, so it is part of the label.** Thirty centimetres is the IPCC
default for mineral soil, and for mineral soil it captures most of the profile's carbon. For
peat it does not. Tropical peat can run several metres deep, so a fixed 30 cm window samples
only a small upper slice of it. A peatland AOI will therefore still report far below its real
soil stock, just less far below than reporting no soil carbon at all.

The component handles this by naming the depth in the narrative rather than saying "soil organic
carbon" unqualified, and by raising a flag when 1.1 reported peatland in the AOI. That check
needs the General stage results; when they are absent the component says so instead of assuming
there is no peat.

**Decisions locked.**

- Extent is every valid pixel inside the AOI. Soil carbon exists under all land cover, so there
  is no forest mask and no land cover filter.
- Nodata inside the AOI is treated as zero carbon, matching 3.1. Coverage is measured and
  flagged below `CARBON_COVERAGE_WARN_PCT`, because with nodata read as zero an incomplete
  raster produces a quiet under-estimate.
- Resampling is `average`, like 3.1, because this is a stock and reprojection has to preserve
  the area weighted mean.
- Not summed with 3.1. See the note at the top of the notebook.

**Example render.**

> **449,000 tCO2e**
>
> The soil in this project area holds approximately 449,000 tCO2e of organic carbon in the top
> 30 cm, an average of 362 tCO2e per hectare.

**Example render, peatland present.**

> **449,000 tCO2e**
>
> The soil in this project area holds approximately 449,000 tCO2e of organic carbon in the top
> 30 cm, an average of 362 tCO2e per hectare. Part of this project area is peatland, where peat
> can extend well below 30 cm, so the real soil carbon stock is likely to be considerably
> higher.

**Narrative not yet specified.** Placeholder wording, written to keep the depth visible.
Replace it once the team settles the phrasing.

**Downstream use.** Soil carbon is the pool most at risk from drainage and conversion on peat,
so it is the quantity behind the Peat Protection pathway. F02-P5 reads it as the stock that
avoided drainage protects.

In [ ]:
PEATLAND_CLASS = 3  # ecosystem code from 1.1


def analyze_soil_organic_carbon(
    aoi: AOI,
    ecosystem_present: set[int] | None = None,
) -> ComponentResult:
    """Component 3.2. Soil organic carbon in the project area, in tCO2e.

    `ecosystem_present` is the Axis 3 set from 1.1. Pass None when the General stage has not
    been run; the peat caveat is then skipped and flagged rather than silently omitted.
    """
    raster = load_raster_clipped(SOIL_CARBON_RASTER, aoi, resampling="average")

    # Nodata inside the AOI counts as zero carbon, matching 3.1. Coverage is measured so an
    # incomplete raster stays visible.
    values = raster.values.filled(0.0).astype(float)

    # tC/ha * ha = tC. No carbon fraction: the raster is already carbon, not biomass.
    soc_tc = float(values.sum()) * raster.pixel_area_ha
    total_tco2e = soc_tc * CO2_PER_C

    if total_tco2e <= 0:
        return not_applicable(
            "3.2 Soil Organic Carbon",
            "No soil carbon data is available for this project area, so soil organic carbon "
            "cannot be estimated.",
        )

    density_tco2e_ha = total_tco2e / aoi.area_ha if aoi.area_ha > 0 else 0.0
    coverage_pct = safe_pct(raster.valid_area_ha, aoi.area_ha)

    flags: list[str] = []
    if coverage_pct < CARBON_COVERAGE_WARN_PCT:
        flags.append(
            f"3.2: the soil carbon raster covers only {coverage_pct:.0f}% of the AOI. Nodata is "
            "counted as zero carbon, so the headline is an under-estimate by an unknown amount."
        )

    # Peat caveat. Depends on 1.1, so the three states are: peat present, no peat, unknown.
    has_peat = None if ecosystem_present is None else PEATLAND_CLASS in ecosystem_present
    peat_clause = ""
    if has_peat is None:
        flags.append(
            "3.2: the General stage has not been run, so the AOI could not be checked for "
            "peatland. If peat is present, this figure understates the soil stock heavily."
        )
    elif has_peat:
        peat_clause = (
            f"Part of this project area is peatland, where peat can extend well below "
            f"{SOIL_CARBON_DEPTH_CM} cm, so the real soil carbon stock is likely to be "
            "considerably higher."
        )
        flags.append(
            f"3.2: peatland present. The raster represents the top {SOIL_CARBON_DEPTH_CM} cm "
            "only, so this is a lower bound on soil carbon, not an estimate of it."
        )

    # Placeholder wording, see the note above.
    narrative = sentences(
        f"The soil in this project area holds approximately {total_tco2e:,.0f} tCO2e of organic "
        f"carbon in the top {SOIL_CARBON_DEPTH_CM} cm, an average of {density_tco2e_ha:,.0f} "
        "tCO2e per hectare.",
        peat_clause,
    )

    return ComponentResult(
        component="3.2 Soil Organic Carbon",
        applicable=True,
        narrative=narrative,
        tables={},
        values={
            "total_tco2e": total_tco2e,          # headline big number
            "density_tco2e_ha": density_tco2e_ha,
            "soc_tc": soc_tc,
            "depth_cm": SOIL_CARBON_DEPTH_CM,
            "coverage_pct": coverage_pct,
            "peatland_present": has_peat,        # True, False, or None when 1.1 is unavailable
        },
        flags=flags,
    )

---
## Shared reading of the WorldClim monthly stack

3.3 and 3.4 both consume twelve monthly rasters and both report the same two things: a twelve
bar chart, and a sentence about the annual figure. The mechanics are shared, so they live in one
helper.

**Data.** WorldClim v`WORLDCLIM_VERSION`, `WORLDCLIM_RESOLUTION` grid, twelve GeoTIFFs per
variable in calendar order.

**This is a 1970-2000 normal, not today's climate.** WorldClim v2.1 averages the three decades
to 2000. For screening a site's bioclimatic setting that is the right kind of data, because a
long normal is what defines which species and forest type belong there. But it is 26 to 56 years
old, and mean temperature across Southeast Asia has risen since that window closed. Every figure
these two components produce carries `WORLDCLIM_PERIOD` in `values` so the frontend can label
it, and the narrative names the period rather than implying current conditions.

**Two different statistics in one card, and the order of operations matters.**

- The **chart** shows twelve monthly values, each the spatial mean over the AOI for that month.
- The **sentence** describes the *annual* figure across space: its minimum, maximum and mean
  over the AOI pixels. For temperature the annual figure is the per pixel mean of twelve months.
  For precipitation it is the per pixel **sum** of twelve months.

For precipitation the sum has to come first, per pixel, and only then the spatial minimum and
maximum. Reversing it gives a wrong range: summing the twelve spatial minima would produce a
value no pixel actually has. The spatial mean happens to survive the swap, because averaging and
summing commute, but the range does not, so the code never takes that shortcut.

**Decisions locked.**

- Resampling is `nearest`, not `bilinear`. These components report a minimum and a maximum, so
  the values must be ones that exist in the source data. Bilinear would invent intermediate
  values and shrink the reported range.
- A pixel counts only when all twelve months are valid. A pixel valid in nine months would
  otherwise produce an annual total of nine months and read as an unusually dry place.
- The chart and the sentence describe the same pixel set, so the twelve monthly means are taken
  over the all-twelve-months-valid pixels too.
- The spatial mean is unweighted. In an equal-area CRS every pixel covers the same ground, so an
  unweighted mean is already area weighted.
- Below `CLIMATE_MIN_PIXELS` valid pixels the range is not reported. A 1 km grid gives a 200 ha
  AOI only a couple of cells, and "ranges from 26.1 to 26.1" is noise dressed as a finding.

**Chart output.** Both components emit their twelve bars in `tables`, as a list of
`MonthlyValue` records with `month`, `label` and `value`. The unit and the axis label travel
next to them in `values` as `chart_unit` and `chart_axis_label`, so the frontend reads the axis
from the data rather than hardcoding one per component. Frozen dataclasses serialise cleanly
through `to_jsonable`, so the series survives the JSON handoff to later stages intact.

In [ ]:
@dataclass(frozen=True)
class MonthlyValue:
    """One bar of a twelve month chart: the spatial mean over the AOI for that month."""

    month: int          # 1 to 12
    label: str          # Jan to Dec
    value: float


@dataclass(frozen=True)
class ClimateStack:
    """Twelve monthly rasters read over the AOI, restricted to fully valid pixels."""

    monthly: list[MonthlyValue]   # spatial mean per month, for the chart
    annual: np.ndarray            # one annual value per valid pixel, 1D
    n_pixels: int

    @property
    def has_range(self) -> bool:
        """Whether a spatial minimum and maximum are worth reporting."""
        return self.n_pixels >= CLIMATE_MIN_PIXELS and self.annual.max() > self.annual.min()


def _read_monthly_stack(paths: list[str], aoi: AOI, annual: str) -> ClimateStack | None:
    """Read twelve monthly rasters and reduce them to a chart series and an annual array.

    `annual` is "mean" for temperature or "sum" for precipitation. It is applied per pixel,
    across the twelve months, before any spatial statistic is taken.

    Returns None when no pixel has all twelve months, so the caller can report not applicable.
    """
    slices = [load_raster_clipped(p, aoi, resampling="nearest") for p in paths]

    shapes = {s.values.shape for s in slices}
    if len(shapes) > 1:
        raise ValueError(
            f"The twelve monthly rasters do not share one grid after clipping: {shapes}. "
            "They must come from the same WorldClim resolution."
        )

    # (12, rows, cols), nodata as NaN so the all-months check is one operation.
    data = np.ma.stack([s.values.astype(float) for s in slices]).filled(np.nan)

    # A pixel counts only when every month is present. See the note above.
    all_valid = ~np.isnan(data).any(axis=0)
    n_pixels = int(all_valid.sum())
    if n_pixels == 0:
        return None

    annual_per_pixel = (
        data.mean(axis=0)[all_valid] if annual == "mean" else data.sum(axis=0)[all_valid]
    )

    # Chart series over the same pixel set, so chart and sentence agree.
    monthly = [
        MonthlyValue(month=m + 1, label=MONTH_LABELS[m], value=float(data[m][all_valid].mean()))
        for m in range(12)
    ]

    return ClimateStack(monthly=monthly, annual=annual_per_pixel, n_pixels=n_pixels)

---
## 3.3 Annual Temperature

Reports the temperature regime of the project area: a twelve month profile as a bar chart, and
the annual mean temperature as a range across the site.

**Data.** `WORLDCLIM_TAVG_RASTERS`, twelve monthly mean temperature rasters in degrees Celsius.

**The annual figure is a per pixel mean of the twelve months**, then summarised across the AOI.

**Example render.**

> Annual mean temperature in the selected area ranges from 24.3 to 26.6 degree Celsius with an
> average of 26.1 degree Celsius.

**Example render, AOI smaller than a few grid cells.**

> Annual mean temperature in the selected area is 26.1 degree Celsius.

**Downstream use.** Temperature bounds which species and forest types are appropriate, so it
feeds species selection in the Activity Catalog alongside elevation from 1.4. It is descriptive
context, not a pathway driver.

In [ ]:
def analyze_annual_temperature(aoi: AOI) -> ComponentResult:
    """Component 3.3. Monthly temperature profile and the annual mean across the AOI."""
    stack = _read_monthly_stack(WORLDCLIM_TAVG_RASTERS, aoi, annual="mean")

    if stack is None:
        return not_applicable(
            "3.3 Annual Temperature",
            "No temperature data is available for this project area.",
        )

    mean_c = float(stack.annual.mean())
    min_c = float(stack.annual.min())
    max_c = float(stack.annual.max())

    flags: list[str] = []
    if stack.has_range:
        narrative = (
            f"Annual mean temperature in the selected area ranges from {min_c:.1f} to "
            f"{max_c:.1f} degree Celsius with an average of {mean_c:.1f} degree Celsius."
        )
    else:
        # Too few cells, or every cell identical. Reporting a range here would be noise.
        narrative = (
            f"Annual mean temperature in the selected area is {mean_c:.1f} degree Celsius."
        )
        flags.append(
            f"3.3: only {stack.n_pixels} climate grid cells fall inside the AOI, so no spatial "
            "range is reported."
        )

    return ComponentResult(
        component="3.3 Annual Temperature",
        applicable=True,
        narrative=narrative,
        tables={"monthly_temperature": stack.monthly},  # twelve bars, one per month
        values={
            # Chart metadata travels with the series so the frontend does not hardcode units.
            "chart_series": "monthly_temperature",
            "chart_unit": "C",
            "chart_axis_label": "Temperature (C)",
            "annual_mean_c": mean_c,
            "annual_min_c": min_c,
            "annual_max_c": max_c,
            "n_pixels": stack.n_pixels,
            "source": f"WorldClim v{WORLDCLIM_VERSION} {WORLDCLIM_RESOLUTION}",
            "period": WORLDCLIM_PERIOD,
        },
        flags=flags,
    )

---
## 3.4 Annual Precipitation

Reports the rainfall regime of the project area: a twelve month profile as a bar chart, and the
annual precipitation total as a range across the site.

**Data.** `WORLDCLIM_PREC_RASTERS`, twelve monthly precipitation rasters in mm.

**The annual figure is a per pixel sum of the twelve months**, then summarised across the AOI.
This is the step that must not be reordered. See the shared note above.

**Example render.**

> Annual precipitation in the selected area ranges from 3,094.0 to 3,268.0 mm with an average of
> 3,149.0 mm.

**Example render, AOI smaller than a few grid cells.**

> Annual precipitation in the selected area is 3,149.0 mm.

**Downstream use.** Rainfall total and its seasonal distribution govern which pathways are
workable: a long dry season raises establishment risk for planting under RESTORE, and links to
the fire and drought hazards in 1.7. The monthly series is kept in `tables` so a later component
can derive dry season length from it rather than re-reading twelve rasters.

In [ ]:
def analyze_annual_precipitation(aoi: AOI) -> ComponentResult:
    """Component 3.4. Monthly rainfall profile and the annual total across the AOI."""
    # "sum": twelve months are added per pixel BEFORE any spatial statistic is taken.
    stack = _read_monthly_stack(WORLDCLIM_PREC_RASTERS, aoi, annual="sum")

    if stack is None:
        return not_applicable(
            "3.4 Annual Precipitation",
            "No precipitation data is available for this project area.",
        )

    mean_mm = float(stack.annual.mean())
    min_mm = float(stack.annual.min())
    max_mm = float(stack.annual.max())

    flags: list[str] = []
    if stack.has_range:
        narrative = (
            f"Annual precipitation in the selected area ranges from {min_mm:,.1f} to "
            f"{max_mm:,.1f} mm with an average of {mean_mm:,.1f} mm."
        )
    else:
        narrative = f"Annual precipitation in the selected area is {mean_mm:,.1f} mm."
        flags.append(
            f"3.4: only {stack.n_pixels} climate grid cells fall inside the AOI, so no spatial "
            "range is reported."
        )

    return ComponentResult(
        component="3.4 Annual Precipitation",
        applicable=True,
        narrative=narrative,
        tables={"monthly_precipitation": stack.monthly},  # twelve bars, one per month
        values={
            # Chart metadata travels with the series so the frontend does not hardcode units.
            "chart_series": "monthly_precipitation",
            "chart_unit": "mm",
            "chart_axis_label": "Precipitation (mm)",
            "annual_mean_mm": mean_mm,
            "annual_min_mm": min_mm,
            "annual_max_mm": max_mm,
            "n_pixels": stack.n_pixels,
            "source": f"WorldClim v{WORLDCLIM_VERSION} {WORLDCLIM_RESOLUTION}",
            "period": WORLDCLIM_PERIOD,
        },
        flags=flags,
    )

---
## Run and save

In [ ]:
# 3.2 reads the Axis 3 ecosystem set from 1.1 to decide whether to add the peat caveat. The
# General stage is optional: without it the caveat is skipped and flagged, and this notebook
# still runs on its own.
try:
    general = load_results(aoi_id, STAGE_GENERAL)
    ecosystem_present = set(component_values(general, "1.1")["present_set"])
except FileNotFoundError:
    ecosystem_present = None
    print("F02-P2 General has not been run for this AOI. The peat check in 3.2 is skipped.")

results: dict[str, ComponentResult] = {}

results["3.1"] = analyze_current_carbon_storage(aoi)
results["3.2"] = analyze_soil_organic_carbon(aoi, ecosystem_present)
results["3.3"] = analyze_annual_temperature(aoi)
results["3.4"] = analyze_annual_precipitation(aoi)

for key, r in results.items():
    print(f"[{key}] {r.component}{'' if r.applicable else '  (not applicable)'}")
    print(f"      {r.narrative}")
    for f in r.flags:
        print(f"      FLAG: {f}")

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_CLIMATE)
print(f"Saved {path}")